# dARK: ciclo de depósito desde DSpace, OJS u otra plataforma

Este notebook simula el punto de vista de una plataforma depositante, no el de un operador de dARK. Sigue un ítem desde su UUID interno hasta su ARK publicado:

1. La plataforma reserva un ARK idempotentemente.
2. Guarda el ARK en el registro local del ítem.
3. Cuando el usuario termina los metadatos y existe una URL pública estable, envía Level 1 y Level 2 al minter.
4. dARK procesa el trabajo en segundo plano: persistencia, réplica y blockchain.
5. La plataforma verifica más tarde que el estado sea `PUBLISHED`.

El mismo patrón sirve para DSpace, OJS, InvenioRDM u otra aplicación: sólo cambia el adaptador que convierte el registro local a Level 1 y el payload original a Level 2.

## Requisitos y variables

El entorno Docker debe tener en ejecución Admin API, Minter API, Store API, Resolver, PostgreSQL, IPFS y los tres workers del minter. En producción la autoridad se aprovisiona previamente; `ENSURE_AUTHORITY=true` es una comodidad para un entorno de prueba.

Variables opcionales: `ADMIN_API_BASE_URL`, `MINTER_BASE_URL`, `RESOLVER_BASE_URL`, `STORE_API_BASE_URL`, `AUTHORITY_ID`, `NAAN`, `PLATFORM_ITEM_ID`, `TARGET_URL`, `POLL_INTERVAL_SECONDS`, `POLL_TIMEOUT_SECONDS` y `ENSURE_AUTHORITY`.

In [ ]:
import os
import time
from pprint import pprint

import requests

ADMIN_API_V1 = f"{os.getenv('ADMIN_API_BASE_URL', 'http://localhost:8000').rstrip('/')}/api/v1/admin"
MINTER_BASE_URL = os.getenv('MINTER_BASE_URL', 'http://localhost:8001').rstrip('/')
MINTER_API_V1 = f"{MINTER_BASE_URL}/api/v1"
RESOLVER_API_V1 = f"{os.getenv('RESOLVER_BASE_URL', 'http://localhost:8002').rstrip('/')}/api/v1"
STORE_API_BASE_URL = os.getenv('STORE_API_BASE_URL', 'http://localhost:8003').rstrip('/')

AUTHORITY_ID = os.getenv('AUTHORITY_ID', f'platform-demo-{int(time.time())}')
NAAN = os.getenv('NAAN', '12345').strip()
PLATFORM_ITEM_ID = os.getenv('PLATFORM_ITEM_ID', f'dspace-item-{int(time.time())}')
TARGET_URL = os.getenv('TARGET_URL', f'https://repository.example.org/items/{PLATFORM_ITEM_ID}')
POLL_INTERVAL_SECONDS = float(os.getenv('POLL_INTERVAL_SECONDS', '2'))
POLL_TIMEOUT_SECONDS = int(os.getenv('POLL_TIMEOUT_SECONDS', '120'))
ENSURE_AUTHORITY = os.getenv('ENSURE_AUTHORITY', 'true').lower() in {'1', 'true', 'yes'}

# En producción, mTLS vincula la conexión a la autoridad. Este header sirve para el perfil local sin mTLS.
MINTER_HEADERS = {'X-Authority-Id': AUTHORITY_ID}

def show(response):
    print('HTTP', response.status_code)
    try:
        pprint(response.json())
    except ValueError:
        print(response.text)

def get_ark(ark):
    response = requests.get(f'{MINTER_API_V1}/arks/{ark}', headers=MINTER_HEADERS, timeout=30)
    response.raise_for_status()
    return response.json()

def wait_until_published(ark):
    deadline = time.time() + POLL_TIMEOUT_SECONDS
    last = None
    while time.time() < deadline:
        last = get_ark(ark)
        print(f"{time.strftime('%H:%M:%S')}  state={last['state']}  l1={last.get('level1_cid')}")
        if last['state'] == 'P':
            return last
        time.sleep(POLL_INTERVAL_SECONDS)
    raise TimeoutError(f'ARK {ark} did not reach PUBLISHED. Last response: {last}')

print('authority:', AUTHORITY_ID)
print('naan:', NAAN)
print('platform item:', PLATFORM_ITEM_ID)
print('target:', TARGET_URL)

## 0. Comprobar que dARK está disponible

La plataforma sólo debe aceptar o reintentar depósitos si puede alcanzar el minter. El estado compacto de workers es barato: no hace consultas de cola, RPC ni Store API.

In [ ]:
minter_health = requests.get(f'{MINTER_BASE_URL}/health', timeout=30)
show(minter_health)
assert minter_health.status_code in {200, 503}

worker_status = requests.get(f'{MINTER_API_V1}/worker/status', timeout=30)
show(worker_status)
assert worker_status.status_code == 200
workers = worker_status.json().get('workers', {})
for worker_name in ('metadata', 'replication', 'chain'):
    assert workers.get(worker_name, {}).get('alive') is True, f'{worker_name} worker is not alive'

## 1. Precondición: autoridad y NAAN

En un servicio real esto es responsabilidad del operador dARK, no del usuario que deposita. La plataforma usa una única identidad de autoridad y nunca recibe la clave blockchain.

In [ ]:
if ENSURE_AUTHORITY:
    created = requests.post(
        f'{ADMIN_API_V1}/authority',
        json={'uuid': AUTHORITY_ID, 'naans': [], 'fund_amount_eth': 0.05},
        timeout=60,
    )
    assert created.status_code in {200, 201, 409}, created.text
    authorized = requests.post(
        f'{ADMIN_API_V1}/authority/{AUTHORITY_ID}/authorize-naan',
        json={'naan': NAAN},
        timeout=60,
    )
    assert authorized.status_code == 200, authorized.text

authority_naans = requests.get(f'{MINTER_API_V1}/authority/{AUTHORITY_ID}/naans', timeout=30)
show(authority_naans)
assert authority_naans.status_code == 200
assert NAAN in authority_naans.json()['naans']

## 2. La plataforma crea el ítem y reserva el ARK

`client_item_id` es el UUID interno de DSpace/OJS. La reserva batch se usa incluso para un único ítem porque aporta idempotencia: ante un timeout, repetir la misma solicitud devuelve el mismo ARK. La plataforma debe guardar ese ARK en su propia base de datos.

In [ ]:
platform_item = {
    'id': PLATFORM_ITEM_ID,
    'workflow_state': 'metadata-entry',
    'dark_ark': None,
}

reserve_payload = {
    'authority_id': AUTHORITY_ID,
    'naan': NAAN,
    'items': [{'client_item_id': platform_item['id']}],
}
reserve = requests.post(f'{MINTER_API_V1}/arks/batch', headers=MINTER_HEADERS, json=reserve_payload, timeout=60)
show(reserve)
assert reserve.status_code in {200, 201}
reservation = reserve.json()['results'][0]
assert reservation['state'] == 'R'
platform_item['dark_ark'] = reservation['ark']
ARK = platform_item['dark_ark']
pprint(platform_item)

## 3. Simular un reintento de red de la plataforma

El segundo `POST /arks/batch` representa un cliente que no recibió la primera respuesta. No debe crear otro identificador.

In [ ]:
reserve_retry = requests.post(f'{MINTER_API_V1}/arks/batch', headers=MINTER_HEADERS, json=reserve_payload, timeout=60)
show(reserve_retry)
assert reserve_retry.status_code in {200, 201}
assert reserve_retry.json()['results'][0]['ark'] == ARK
assert get_ark(ARK)['state'] == 'R'

## 4. El usuario termina los metadatos y la plataforma los transforma

La plataforma conserva su formato original como Level 2 y construye una proyección pública mínima como Level 1. Para DSpace podría ser `oai_dc` XML; para OJS, Crossref/JATS o JSON del artículo. La URL `target` debe ser una landing page pública y estable, normalmente el Handle o URL canónica del artículo.

In [ ]:
platform_item.update({
    'workflow_state': 'ready-for-publication',
    'title': 'Objeto de demostración desde una plataforma depositante',
    'authors': ['Autor/a de ejemplo'],
    'year': 2026,
})

level1 = {
    'title': platform_item['title'],
    'authors': platform_item['authors'],
    'year': platform_item['year'],
    'publisher': 'Repositorio de demostración',
    'resource_type': 'article',
    'language': 'es',
    'abstract': 'Registro creado por la simulación de integración dARK.',
    'subjects': ['interoperabilidad', 'identificadores persistentes'],
    'alternate_identifiers': [{'schema': 'platform-item-id', 'value': platform_item['id']}],
    'alternate_urls': [TARGET_URL],
}
level2_oai_dc = f'''<oai_dc:dc xmlns:oai_dc="http://www.openarchives.org/OAI/2.0/oai_dc/" xmlns:dc="http://purl.org/dc/elements/1.1/">
  <dc:identifier>{TARGET_URL}</dc:identifier>
  <dc:identifier>{platform_item['id']}</dc:identifier>
  <dc:title>{platform_item['title']}</dc:title>
  <dc:creator>{platform_item['authors'][0]}</dc:creator>
  <dc:date>{platform_item['year']}</dc:date>
  <dc:type>Article</dc:type>
</oai_dc:dc>'''.strip()

pprint(level1)

## 5. Registrar metadata: `RESERVED → DRAFT`

Esta llamada sólo confirma que dARK ha aceptado los metadatos en PostgreSQL. No espera IPFS ni blockchain; por eso la plataforma puede devolver control al usuario inmediatamente.

In [ ]:
stage = requests.put(
    f'{MINTER_API_V1}/arks/{ARK}',
    headers=MINTER_HEADERS,
    json={
        'authority_id': AUTHORITY_ID,
        'target': TARGET_URL,
        'minimal_metadata': level1,
        'original_metadata': level2_oai_dc,
        'metadata_schema': 'oai_dc',
        'metadata_media_type': 'application/xml',
    },
    timeout=60,
)
show(stage)
assert stage.status_code == 200
assert stage.json()['state'] == 'D'
assert get_ark(ARK)['state'] == 'D'
platform_item['workflow_state'] = 'waiting-for-dark-publication'
pprint(platform_item)

## 6. Esperar y verificar la publicación de dARK

A partir de aquí no se dispara una llamada de publicación desde DSpace/OJS. Los workers del minter realizan el trabajo. La plataforma puede ejecutar esta celda como tarea diferida, con backoff, hasta observar `P`. Si se alcanza el timeout, el ARK sigue siendo recuperable; revisar `/api/v1/worker/errors?list=all&authority_id=...` y el estado de workers.

In [ ]:
published = wait_until_published(ARK)
pprint(published)
assert published['state'] == 'P'
assert published['target'] == TARGET_URL
assert published.get('level1_cid'), 'Published ARK must expose a Level-1 CID'
assert published.get('level2_cid'), 'Published ARK must expose a Level-2 CID'
platform_item['workflow_state'] = 'dark-published'
platform_item['dark_level1_cid'] = published['level1_cid']
pprint(platform_item)

## 7. Verificar almacenamiento, resolución y trazabilidad

`PUBLISHED` confirma publicación en cadena. Esta comprobación adicional muestra las observaciones de réplica de Store API y valida que el resolver redirige al usuario a la URL de la plataforma.

In [ ]:
for label, cid in [('level1', published['level1_cid']), ('level2', published['level2_cid'])]:
    replication = requests.get(f'{STORE_API_BASE_URL}/v1/status/{cid}', timeout=30)
    print(label, cid)
    show(replication)
    assert replication.status_code == 200

resolved = requests.get(f'{RESOLVER_API_V1}/arks/{ARK}', allow_redirects=False, timeout=30)
show(resolved)
assert resolved.status_code in {302, 307}
assert resolved.headers['location'] == TARGET_URL

print(f'Integración completada: {platform_item["id"]} -> {ARK} -> {TARGET_URL}')

## Patrón para llevar a producción

- Use el UUID inmutable del ítem como `client_item_id`; no reserve de nuevo con un ID distinto tras un error de red.
- Guarde el ARK retornado en la plataforma antes de mostrarlo al usuario.
- Envíe `PUT` sólo cuando tenga metadata válida y una URL pública estable.
- Trate `DRAFT` como una aceptación asíncrona, no como publicación.
- Consulte `GET /arks/{ark}` desde una tarea diferida hasta `PUBLISHED`; no bloquee la solicitud web del usuario.
- Use mTLS en producción y deje las claves de autoridad exclusivamente en dARK.
- Si un ARK permanece en `DRAFT`/`UPDATE`, el operador debe revisar workers y `/worker/errors`; la API actual no expone aún el detalle de error directamente en la respuesta individual del ARK.